# Water Bottle Demand Forecasting Pipeline

This notebook contains the end-to-end exploratory data analysis and predictive machine learning modeling required to optimize supply and prevent over/under-manufacturing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Set visualization style for charts
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the dataset.
# Prefer the ENRICHED table (weather + climate + geography + political) built by
# `python run_pipeline.py`; fall back to the raw provided Excel file.
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
enriched = ROOT / "data" / "processed" / "enriched_demand.csv"
raw = ROOT / "data" / "raw" / "Water_Bottle_Demand_Dataset.xlsx"

if enriched.exists():
    df = pd.read_csv(enriched)
    print(f"Loaded ENRICHED dataset: {df.shape[0]} rows x {df.shape[1]} cols")
else:
    df = pd.read_excel(raw)
    print(f"Loaded RAW dataset: {df.shape[0]} rows x {df.shape[1]} cols "
          "(run `python run_pipeline.py` to build the enriched version)")

df['Date'] = pd.to_datetime(df['Date'])
display(df.head())

## 1. Exploratory Data Analysis (EDA)
Visualizing the relationships between weather, seasonality, and demand.

In [ ]:
# Time-series plot of demand
plt.figure(figsize=(15, 6))
sns.lineplot(data=df.sample(1000), x='Date', y='Target_Demand', hue='Region', alpha=0.7)
plt.title('Sampled Daily Demand Trends Over Time by Region')
plt.ylabel('Demand (Units)')
plt.show()

In [ ]:
# Heatmap for Feature Correlation
plt.figure(figsize=(8, 6))
numeric_cols = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
# Temperature vs Demand Scatter Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df.sample(1000), x='Temperature_C', y='Target_Demand', hue='Season', alpha=0.8)
plt.title('Impact of Temperature on Water Bottle Demand')
plt.show()

## 2. Feature Engineering
Extracting temporal features and encoding categorical data for the algorithm.

In [ ]:
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Year'] = df['Date'].dt.year

# One-hot encode the categorical drivers that are present.
cat_cols = [c for c in ['Region', 'Season', 'Climate_Zone'] if c in df.columns]
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Drop identifiers / leakage columns, keep only numeric predictors.
drop_cols = [c for c in ['Date', 'Target_Demand', 'Base_Demand', 'City_Proxy']
             if c in df_encoded.columns]
X = df_encoded.drop(columns=drop_cols).select_dtypes(include=[np.number, 'bool']).astype(float)
y = df_encoded['Target_Demand']

# Splitting the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training Data Shape: {X_train.shape}")
print("\nNote: this notebook is the quick exploratory baseline. The production "
      "pipeline (src/) adds time-based validation, model selection and future "
      "forecasting — run `python run_pipeline.py`.")

## 3. Predictive Modeling
Training a Random Forest Regressor to forecast next period demand.

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred):.2f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

In [ ]:
# Feature Importance Visualization
importance = pd.DataFrame({'Feature': X.columns, 'Importance': model.feature_importances_})
importance = importance.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance, x='Importance', y='Feature', palette='mako')
plt.title('Feature Importance for Demand Prediction')
plt.show()